# Notebook 4: A neural network from scratch (NumPy only)

Step 4 of the ladder. Notebook 2 ended on a failure: **one neuron can only draw a straight line**, so it scored 51% on the XOR-style problem. The fix was a hand-made extra input (`x1*x2`). Here we let the network **learn its own extra inputs** by stacking neurons in layers. Then we open the hood on everything Keras was hiding in `model.fit` and `train_on_batch`: the forward pass, the backward pass, and the update.

You need: neurons and cross-entropy from Notebook 2, gradient descent from Notebook 1, and the train/validation idea from Notebook 3. NumPy and matplotlib only; runs in well under a minute.

### What you'll be able to say at the end
- A **layer** is many neurons side by side; a **network** is layers feeding layers. The hidden layer builds new inputs for the next layer.
- The **backward pass** is the chain rule in disguise: "blame travels backward, scaled by each weight". You can check it by nudging, as before.
- Where a network's **many knobs** come from: width, depth, activation, initialization, and what each one breaks at its extremes.
- What "1 billion parameters" counts, and how to count your own.
- Why Keras's `Dense`, `train_on_batch` and `fit` are exactly this machine.

Same markers as before: `WHAT / WHY / SAME AS / BECOMES / WATCH` comments, **PREDICT** cells, a knob ledger, and a sandbox.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# WHAT: plot styling only (same quiet look as Notebooks 1 to 3). Nothing here affects the math.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, MUTED, GRIDC = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 11, "axes.labelsize": 10, "font.size": 10,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

def tag(ax, x, y, text, color, dx=4, dy=0, ha="left"):
    # WHAT: write a small label next to a line instead of making you decode a legend.
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", color=color, fontsize=9, va="center", ha=ha)

def sigmoid(z):
    # SAME AS: Notebook 2. Squash any number into 0..1.
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def make_xor(n, seed):
    # SAME AS: Notebook 2. A part passes when the two readings have the SAME sign.
    r = np.random.default_rng(seed)
    X = r.uniform(-2, 2, (n, 2))
    return X, (X[:, 0] * X[:, 1] > 0).astype(float)

print("ready")

ready


# Part 1: Stack two layers by hand (no learning yet)

> **You are here:** you'll *set the weights by hand* to see that a two-layer network **can** solve XOR, before asking gradient descent to find the weights itself.

The idea: XOR passes when the signs agree. That's "both readings high" **or** "both readings low". Each of those is a straight-line question that one neuron *can* answer:
- **Hidden neuron A** asks: "are both high?" Fires when `x1 + x2` is well above 1.
- **Hidden neuron B** asks: "are both low?" Fires when `x1 + x2` is well below -1.
- **Output neuron** asks: "did A *or* B fire?"

The hidden layer's outputs become the output neuron's *inputs*. That's the whole trick. Check the four corner cases (readings of +1 or -1):

In [2]:
def hand_net(x1, x2):
    # WHAT: layer 1: two neurons, each a weighted sum + squash, exactly Notebook 2's neuron.
    hA = sigmoid(5 * (x1 + x2 - 1))        # WHAT: "both high": z is +5 only when x1 = x2 = +1
    hB = sigmoid(5 * (-x1 - x2 - 1))       # WHAT: "both low":  z is +5 only when x1 = x2 = -1
    # WHAT: layer 2: ONE neuron whose inputs are (hA, hB). "A or B": positive weights, bias that needs at least one to fire.
    return hA, hB, sigmoid(10 * (hA + hB) - 5)

print("  x1   x2 | hidden A  hidden B | p(pass) | truth")
for x1, x2 in [(1, 1), (-1, -1), (1, -1), (-1, 1)]:
    hA, hB, p = hand_net(x1, x2)
    print(f" {x1:+3d}  {x2:+3d} |  {hA:7.3f}  {hB:7.3f} | {p:7.3f} | {'pass' if x1 * x2 > 0 else 'fail'}")

  x1   x2 | hidden A  hidden B | p(pass) | truth
  +1   +1 |    0.993    0.000 |   0.993 | pass
  -1   -1 |    0.000    0.993 |   0.993 | pass
  +1   -1 |    0.007    0.007 |   0.008 | fail
  -1   +1 |    0.007    0.007 |   0.008 | fail


**What to notice.** Hidden A lights up only for (+1,+1), hidden B only for (-1,-1), and the output says "pass" when either does. **No single straight line separates the passes from the fails in the original `(x1, x2)` picture, but the hidden layer re-describes each part as `(A, B)`, and in *that* picture a straight line works.** That's what "learning its own inputs" means.

I picked those weights by hand because I knew the answer. In real problems nobody does. The weights get found by gradient descent, which needs the slope of the loss for every weight. Computing those slopes is the **backward pass**.

# Part 2: The forward pass, in matrix form

> **You are here:** write the hand-built network as general code, so weights are arrays and it handles many parts at once.

For a batch of `n` parts with `d` readings each (shape `n x d`), and a hidden layer of `h` neurons:
- `W1` has shape `d x h` (one column of weights per hidden neuron), `b1` has `h` numbers.
- `z1 = X @ W1 + b1` is every hidden neuron's weighted sum for every part (shape `n x h`). `a1 = sigmoid(z1)`.
- `W2` is `h x 1`, `b2` one number. `p = sigmoid(a1 @ W2 + b2)`.

**One line of NumPy (`X @ W1`) does all the neurons for all the parts.** That's why neural nets are written in matrices: it's the same neuron from Notebook 2, thousands of times at once.

In [3]:
def init_params(d, h, seed, scale=1.0):
    r = np.random.default_rng(seed)
    # WHAT: start with small RANDOM weights. (Part 6 shows why they cannot all start equal, and why the size matters.)
    return {"W1": r.normal(0, scale / np.sqrt(d), (d, h)), "b1": np.zeros(h),
            "W2": r.normal(0, scale / np.sqrt(h), (h, 1)), "b2": np.zeros(1)}

def forward2(P, X):
    a1 = sigmoid(X @ P["W1"] + P["b1"])            # WHAT: hidden layer: every hidden neuron, every part, in one line
    p = sigmoid(a1 @ P["W2"] + P["b2"])[:, 0]      # WHAT: output neuron: its inputs are the hidden layer's outputs
    return a1, p

def ce_loss(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))     # SAME AS: Notebook 2's cross-entropy

# shapes for 3 parts, 2 readings, 4 hidden neurons
P0 = init_params(2, 4, seed=0)
Xdemo = np.array([[1.0, 1.0], [-1.0, -1.0], [1.0, -1.0]])
a1, p = forward2(P0, Xdemo)
print("X  shape:", Xdemo.shape, "  W1:", P0["W1"].shape, "  hidden a1:", a1.shape, "  W2:", P0["W2"].shape, "  output p:", p.shape)
print("parameters:", sum(v.size for v in P0.values()), " = (2*4 + 4) in layer 1 + (4*1 + 1) in layer 2")

X  shape: (3, 2)   W1: (2, 4)   hidden a1: (3, 4)   W2: (4, 1)   output p: (3,)
parameters: 17  = (2*4 + 4) in layer 1 + (4*1 + 1) in layer 2


# Part 3: The backward pass (the chain rule, in words)

> **You are here:** the loss is a number built from `p`, which is built from `a1`, which is built from the weights. To nudge every weight downhill we need `d(loss)/d(each weight)`.

The chain rule says: *to find how the loss changes when you wiggle a weight deep inside, multiply the sensitivities along the path from that weight to the loss.* In practice you walk **backwards** from the loss, carrying one "blame" number per neuron:

1. **Output neuron.** From Notebook 2: the blame on its weighted sum is `(p - y)` (spread over the `n` parts by dividing by `n`). Call it `dz2`.
2. **Output weights.** `dW2 = a1.T @ dz2` (each weight's blame = the blame times the input that weight saw: *same sentence as Notebook 2*). `db2 = sum(dz2)`.
3. **Pass blame back.** The hidden neurons share the blame in proportion to the weight connecting them to the output: `da1 = dz2 @ W2.T`.
4. **Through each hidden squash.** Multiply by the squash's own slope: `dz1 = da1 * a1 * (1 - a1)` (the sigmoid's slope in terms of its output).
5. **First-layer weights.** `dW1 = X.T @ dz1`, `db1 = sum(dz1)`.

Every step is a line you already know. **Backpropagation is just this walk, written once.** Then check it by nudging, as always:

In [4]:
def backward2(P, X, y):
    a1, p = forward2(P, X)
    n = len(y)
    dz2 = ((p - y) / n)[:, None]                  # WHAT: step 1: blame on the output's weighted sum (Notebook 2's (p - y), averaged)
    dW2 = a1.T @ dz2                              # WHAT: step 2: (blame) x (the input that weight saw)
    db2 = dz2.sum(0)
    da1 = dz2 @ P["W2"].T                         # WHAT: step 3: blame flows back along the weights
    dz1 = da1 * a1 * (1 - a1)                     # WHAT: step 4: ... through the hidden squash's slope
    dW1 = X.T @ dz1                               # WHAT: step 5: same "blame x input" rule for the first layer
    db1 = dz1.sum(0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

def numeric_grad(P, X, y, h=1e-6):
    # WHAT: nudge every single parameter up and down, one at a time. Slow, but needs no formulas: the ground truth.
    out = {}
    for k, v in P.items():
        g = np.zeros_like(v)
        for idx in np.ndindex(v.shape):
            old = v[idx]
            v[idx] = old + h; up = ce_loss(forward2(P, X)[1], y)
            v[idx] = old - h; down = ce_loss(forward2(P, X)[1], y)
            v[idx] = old
            g[idx] = (up - down) / (2 * h)
        out[k] = g
    return out

Xc, yc = make_xor(30, seed=5)
Pc = init_params(2, 4, seed=1)
mine, nudged = backward2(Pc, Xc, yc), numeric_grad(Pc, Xc, yc)
for k in Pc:
    print(f"{k}: biggest difference between my backward pass and nudging = {np.max(np.abs(mine[k] - nudged[k])):.2e}")
print("parameters checked:", sum(v.size for v in Pc.values()))

W1: biggest difference between my backward pass and nudging = 6.50e-11
b1: biggest difference between my backward pass and nudging = 8.89e-11
W2: biggest difference between my backward pass and nudging = 8.81e-11
b2: biggest difference between my backward pass and nudging = 1.01e-10
parameters checked: 17


**What to notice.** Every one of the 17 slopes agrees with nudging to about 10 decimal places. That check is the single most useful debugging habit in this subject: **if a hand-written backward pass ever disagrees with nudging, the backward pass is wrong.** (Libraries like Keras do this walk automatically for any network. That's *automatic differentiation*, and the reason you never write it by hand in practice. You wrote it once so it isn't magic.)

# Part 4: Train it on XOR

> **You are here:** forward pass, backward pass, and Notebook 1's update: the full training loop.

**PREDICT first.** One neuron got 51% on this data. With 4 hidden neurons, what accuracy do you expect on held-out data: 60%, 80%, or above 95%? And **how many steps** do you think it takes: tens, thousands, or a million? Type a guess.

In [5]:
# PREDICT: held-out accuracy with 4 hidden neurons? and roughly how many gradient steps?
my_prediction = ""
print("your prediction:", my_prediction if my_prediction else "(blank: type one above first)")

your prediction: (blank: type one above first)


In [6]:
Xtr, ytr = make_xor(300, seed=11); Xte, yte = make_xor(300, seed=12)      # SAME AS: Notebook 2's XOR data (same seeds)

P = init_params(2, 4, seed=0)
lr = 1.0                                    # WHAT: learning rate (search knob). Larger than before, because the squash flattens gradients.
losses, test_acc = [], []
print(" step | train loss | held-out accuracy")
for step in range(0, 6001):
    if step % 50 == 0:
        losses.append(ce_loss(forward2(P, Xtr)[1], ytr))
        test_acc.append(np.mean((forward2(P, Xte)[1] > 0.5) == (yte == 1)))
    if step in (0, 100, 500, 1000, 2000, 4000, 6000):
        print(f" {step:5d} |   {losses[-1]:.4f}   |   {test_acc[-1]:.1%}")
    g = backward2(P, Xtr, ytr)
    for k in P:
        P[k] = P[k] - lr * g[k]             # SAME AS: Notebook 1's one-line update, now applied to EVERY weight in both layers

 step | train loss | held-out accuracy
     0 |   0.7441   |   53.0%
   100 |   0.6872   |   58.7%
   500 |   0.3442   |   82.7%
  1000 |   0.2664   |   86.0%


  2000 |   0.1048   |   96.0%


  4000 |   0.0750   |   96.3%


  6000 |   0.0643   |   96.3%


In [7]:
fig, (a1_, a2_) = plt.subplots(1, 2, figsize=(11.5, 4.4))
ax = a1_
ax.scatter(*Xte[yte == 1].T, s=18, color=BLUE, marker="o"); ax.scatter(*Xte[yte == 0].T, s=22, color=ORANGE, marker="^")
g1, g2 = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-2, 2, 200))
gp = forward2(P, np.column_stack([g1.ravel(), g2.ravel()]))[1].reshape(g1.shape)
ax.contour(g1, g2, gp, levels=[0.5], colors=[INK], linewidths=1.6)
ax.set_xlabel("reading x1"); ax.set_ylabel("reading x2"); ax.set_title("The learned boundary bends around the corners")
steps_axis = np.arange(len(losses)) * 50
a2_.plot(steps_axis, losses, color=BLUE)
a2_.set_xlabel("step"); a2_.set_ylabel("train loss"); a2_.set_title("Loss: a plateau near 0.34, then a second drop")
plt.tight_layout(); plt.show()

<Figure size 1265x484 with 2 Axes>

**What to notice.**
- Held-out accuracy went from about 53% to about 96%, on the same data where one neuron got 51%. The boundary on the left has **bent** around the corners: a curve a single neuron can never draw.
- It took about 2,000 steps, not tens and not a million. The loss curve has a **plateau** near 0.34 (around steps 400 to 800) before a second drop: gradient descent can sit on flat stretches for a while before the hidden neurons "find" their roles.
- It reaches about 96%, not 100%, with this many steps and this width. (Notebook 2's hand-made `x1*x2` input got 99%: giving the right feature beats learning it from scratch when you know it. The network pays for needing no hint.)
- This run used random start number 0. **That choice matters**, as the next table shows.

# Part 5: The width knob, and the knob nobody mentions (the random start)

> **You are here:** the hidden width (how many neurons) is the first *model* knob. Test it, and see something unsettling about one run vs many.

First, a general network class that handles any number of layers and any squash. It's the same maths as `forward2`/`backward2`, written once for any depth, so we can try depth and activations later. We verify it matches the hand-written version before trusting it.

In [8]:
ACT = {   # WHAT: each squash and its slope written in terms of its OUTPUT a (what the backward pass has on hand)
    "sigmoid": (lambda z: 1 / (1 + np.exp(-np.clip(z, -500, 500))), lambda a: a * (1 - a)),
    "tanh":    (np.tanh,                                           lambda a: 1 - a ** 2),
    "relu":    (lambda z: np.maximum(z, 0),                        lambda a: (a > 0).astype(float)),
}

def init_net(sizes, seed, act="tanh", scale=1.0):
    # WHAT: one weight matrix + bias vector per layer. sizes = [inputs, hidden..., 1].  scale is the init-size knob (Part 6).
    r = np.random.default_rng(seed)
    gain = 2.0 if act == "relu" else 1.0                         # WHAT: relu throws away half its signal, so it gets a bigger start
    return [(r.normal(0, scale * np.sqrt(gain / a), (a, b)), np.zeros(b)) for a, b in zip(sizes[:-1], sizes[1:])]

def net_forward(net, X, act):
    f, _ = ACT[act]
    acts = [X]
    for W, b in net[:-1]:
        acts.append(f(acts[-1] @ W + b))                         # WHAT: hidden layers: weighted sum, then squash
    W, b = net[-1]
    p = sigmoid(acts[-1] @ W + b)[:, 0]                          # WHAT: the last layer is always a sigmoid neuron (Notebook 2)
    return acts, p

def net_backward(net, X, y, act, l2=0.0):
    _, df = ACT[act]
    acts, p = net_forward(net, X, act)
    dz = ((p - y) / len(y))[:, None]                             # WHAT: start the blame at the output, as in Part 3
    grads = [None] * len(net)
    for i in reversed(range(len(net))):
        W, _ = net[i]
        grads[i] = (acts[i].T @ dz + 2 * l2 * W, dz.sum(0))      # WHAT: (blame) x (input seen); l2 adds Notebook 3's leash on the weights
        if i > 0:
            dz = (dz @ W.T) * df(acts[i])                        # WHAT: pass blame back along the weights, then through the squash slope
    return grads

def count_params(sizes):
    return sum(a * b + b for a, b in zip(sizes[:-1], sizes[1:]))

# VERIFY: same weights in both implementations -> same slopes?
Pw = init_params(2, 4, seed=3)
net_same = [(Pw["W1"].copy(), Pw["b1"].copy()), (Pw["W2"].copy(), Pw["b2"].copy())]
g_old = backward2(Pw, Xc, yc); g_new = net_backward(net_same, Xc, yc, "sigmoid")
print("generic vs hand-written backward pass, biggest difference:",
      max(np.max(np.abs(g_old["W1"] - g_new[0][0])), np.max(np.abs(g_old["W2"] - g_new[1][0]))))
# VERIFY: and the generic one passes the nudging check for a deeper, different-activation net
net_deep = init_net([2, 5, 4, 1], seed=2, act="tanh")
def num_check(net, act, h=1e-6):
    worst = 0
    ana = net_backward(net, Xc, yc, act)
    for li, (W, b) in enumerate(net):
        for arr, which in ((W, 0), (b, 1)):
            for idx in np.ndindex(arr.shape):
                old = arr[idx]
                arr[idx] = old + h; up = ce_loss(net_forward(net, Xc, act)[1], yc)
                arr[idx] = old - h; dn = ce_loss(net_forward(net, Xc, act)[1], yc)
                arr[idx] = old
                worst = max(worst, abs((up - dn) / (2 * h) - ana[li][which][idx]))
    return worst
print(f"nudging check on a 2-5-4-1 tanh network ({count_params([2, 5, 4, 1])} parameters): biggest difference {num_check(net_deep, 'tanh'):.2e}")

generic vs hand-written backward pass, biggest difference: 3.469446951953614e-18
nudging check on a 2-5-4-1 tanh network (44 parameters): biggest difference 2.37e-10


In [9]:
def standardize(X_tr):
    mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-9      # SAME AS: Notebook 1, Part 5: centered/scaled inputs train faster and safer
    return mu, sd

def train_net(X, y, hidden=(8,), act="tanh", lr=0.5, steps=3000, l2=0.0, seed=0, scale=1.0, Xval=None, yval=None, every=100):
    # WHAT: the whole training loop for any network: forward, backward, update, repeat. Full-batch gradient descent.
    mu, sd = standardize(X)
    Z = (X - mu) / sd
    net = init_net([X.shape[1], *hidden, 1], seed, act, scale)
    hist = []
    for step in range(steps + 1):
        if Xval is not None and step % every == 0:
            hist.append((step, ce_loss(net_forward(net, Z, act)[1], y), ce_loss(net_forward(net, (Xval - mu) / sd, act)[1], yval)))
        grads = net_backward(net, Z, y, act, l2)
        net = [(W - lr * gW, b - lr * gb) for (W, b), (gW, gb) in zip(net, grads)]
    predict = lambda Xn: net_forward(net, (Xn - mu) / sd, act)[1]
    return predict, net, hist

def acc_of(predict, X, y):
    return float(np.mean((predict(X) > 0.5) == (y == 1)))

print(" hidden neurons | parameters | held-out accuracy per random start (10 starts, sorted) | solved (>= 95%)")
width_results = {}
for h in (1, 2, 4, 8, 16):
    accs = sorted(acc_of(train_net(Xtr, ytr, (h,), "sigmoid", lr=1.0, steps=3000, seed=s)[0], Xte, yte) for s in range(10))
    width_results[h] = accs
    print(f"     {h:2d}         |    {count_params([2, h, 1]):3d}     | " + " ".join(f"{a:.2f}" for a in accs) + f" |   {sum(a >= 0.95 for a in accs)}/10")

 hidden neurons | parameters | held-out accuracy per random start (10 starts, sorted) | solved (>= 95%)


      1         |      5     | 0.65 0.67 0.68 0.68 0.68 0.68 0.68 0.68 0.70 0.71 |   0/10


      2         |      9     | 0.82 0.82 0.82 0.82 0.82 0.82 0.89 0.89 0.89 0.89 |   0/10


      4         |     17     | 0.89 0.92 0.94 0.94 0.96 0.96 0.96 0.97 0.97 0.97 |   6/10


      8         |     33     | 0.96 0.96 0.96 0.97 0.97 0.97 0.97 0.97 0.98 0.98 |   10/10


     16         |     65     | 0.96 0.96 0.96 0.97 0.97 0.97 0.97 0.97 0.98 0.98 |   10/10


**What to notice** (read the table, not just the headline):
- **1 hidden neuron** never solves it (65% to 71% held-out). A single hidden neuron is a squashed straight line feeding a final squash, so the boundary is still one straight line. A straight line can score above 50% here by cutting off one corner, but it can never get all four right.
- **2 hidden neurons** *can* solve it (Part 1 did, by hand), but gradient descent from these 10 starts found only 82% to 89% in 3,000 steps. Being *able* to represent a solution doesn't mean the search will find it.
- **4 hidden neurons** solved it from 6 of 10 starts and landed between 89% and 94% on the other four. **8 and 16** solved it from all 10. The same width can succeed from one random start and fall short from another, so the random start is a knob too (an *experience/search* one) that no model card lists. Judging a design from a single run can mislead you in either direction.
- 16 neurons did no better than 8 here, so extra width past what the problem needs buys little. It does cost more parameters and compute.

# Part 6: Initialization: why the start matters

> **You are here:** you saw that random starts decide success. Two ways to start badly, shown on purpose.

**Bad start 1: all weights equal (e.g. all zero).** Every hidden neuron then computes the *same* thing and receives the *same* blame, so they stay identical forever, and a 4-neuron layer is really 1 neuron. **Bad start 2: weights too big or too small.** Too small, and signals and slopes shrink to nothing as they pass through layers. Too big, and the squash saturates (flat at 0 or 1), where its slope is nearly zero.

In [10]:
# Bad start 1: all-zero weights
Zs = (Xtr - Xtr.mean(0)) / Xtr.std(0)
net0 = init_net([2, 4, 1], seed=0, act="sigmoid", scale=0.0)           # WHAT: scale = 0 makes every weight exactly 0
for _ in range(2000):
    g = net_backward(net0, Zs, ytr, "sigmoid")
    net0 = [(W - 1.0 * gW, b - 1.0 * gb) for (W, b), (gW, gb) in zip(net0, g)]
W1z, W2z = net0[0][0], net0[1][0]
print("all-zero start, after 2000 steps:")
print("   first-layer weights moved (biggest |weight|):", round(float(np.max(np.abs(W1z))), 4))
print("   but do the four hidden neurons differ?  biggest difference between their weight columns:", float(np.max(np.abs(W1z - W1z[:, [0]]))))
print("   second-layer weights                :", np.round(W2z[:, 0], 4), "  (all four identical)")
print(f"   held-out accuracy                   : {np.mean((net_forward(net0, (Xte - Xtr.mean(0)) / Xtr.std(0), 'sigmoid')[1] > 0.5) == (yte == 1)):.1%}")

# Bad start 2: the SIZE of the random start
print()
print(" init scale | median final train loss | median held-out accuracy   (10 starts, 2000 steps, 4 hidden neurons, sigmoid)")
scale_results = {}
for sc in (0.01, 0.1, 1.0, 3.0, 10.0):
    res = []
    for s in range(10):
        pred, _, _ = train_net(Xtr, ytr, (4,), "sigmoid", lr=1.0, steps=2000, seed=s, scale=sc)
        res.append((ce_loss(pred(Xtr), ytr), acc_of(pred, Xte, yte)))
    scale_results[sc] = (float(np.median([r[0] for r in res])), float(np.median([r[1] for r in res])))
    print(f"   {sc:6.2f}   |        {scale_results[sc][0]:.3f}            |        {scale_results[sc][1]:.1%}")

all-zero start, after 2000 steps:
   first-layer weights moved (biggest |weight|): 0.6704
   but do the four hidden neurons differ?  biggest difference between their weight columns: 0.0
   second-layer weights                : [0.4038 0.4038 0.4038 0.4038]   (all four identical)
   held-out accuracy                   : 59.3%

 init scale | median final train loss | median held-out accuracy   (10 starts, 2000 steps, 4 hidden neurons, sigmoid)


     0.01   |        0.335            |        82.7%


     0.10   |        0.331            |        82.7%


     1.00   |        0.100            |        95.8%


     3.00   |        0.087            |        96.0%


    10.00   |        0.102            |        94.3%


**What to notice.**
- **All-zero start:** the first layer's weights *do* move (once the output weights become nonzero), but the four hidden neurons stay **exactly identical** (the difference printed is 0.0), and the output weights are four copies of one number. Four neurons are doing the work of one, which is why accuracy stays low (about 59%, like the one-hidden-neuron case in Part 5). Random starts exist to **break the symmetry** so that each hidden neuron can learn a different job.
- **Size of the random start:** with tiny weights (0.01 and 0.1) the network was stuck at about 83% after 2,000 steps (loss about 0.33, the same plateau as before; I did not run longer to see whether it escapes). A scale of 1 or 3 reached about 96%. A scale of 10 reached 94%, only slightly worse, so on this small network "too big" barely hurt. The harm from big or tiny starts grows with **depth**, which is what Part 7 shows.

# Part 7: The squash function (activation) and depth

> **You are here:** the hidden-layer squash is a *model* knob. Sigmoid is the one we've used; two others dominate in practice. **tanh** is a sigmoid stretched to run from -1 to +1 (centered on zero). **ReLU** is `max(0, z)`: zero for negatives, a straight line for positives. It is flat on one side only, and cheap.

Compare them on the XOR problem (same width, same steps, 10 random starts each), then look at what depth does to the slopes.

In [11]:
print(" activation | median final train loss | median held-out accuracy | starts solved (>= 95%)")
act_results = {}
for act, lr_ in (("sigmoid", 1.0), ("tanh", 0.5), ("relu", 0.3)):
    res = []
    for s in range(10):
        pred, _, _ = train_net(Xtr, ytr, (8,), act, lr=lr_, steps=2000, seed=s)
        res.append((ce_loss(pred(Xtr), ytr), acc_of(pred, Xte, yte)))
    act_results[act] = (float(np.median([r[0] for r in res])), float(np.median([r[1] for r in res])), sum(r[1] >= 0.95 for r in res))
    print(f"  {act:8s}  |        {act_results[act][0]:.3f}            |         {act_results[act][1]:.1%}          |   {act_results[act][2]}/10   (lr {lr_})")
print("(learning rates differ per activation: I picked each from a quick try, so this is a rough comparison, not a ranking)")

 activation | median final train loss | median held-out accuracy | starts solved (>= 95%)


  sigmoid   |        0.075            |         96.8%          |   10/10   (lr 1.0)


  tanh      |        0.050            |         97.3%          |   10/10   (lr 0.5)


  relu      |        0.029            |         98.7%          |   10/10   (lr 0.3)
(learning rates differ per activation: I picked each from a quick try, so this is a rough comparison, not a ranking)


In [12]:
def layer_grad_sizes(act, depth=6, width=8, seed=0):
    # WHAT: build a deep net, run ONE forward+backward pass at the start, and report how big the gradient is at each layer.
    net = init_net([2] + [width] * depth + [1], seed, act)
    g = net_backward(net, Zs, ytr, act)
    return [float(np.sqrt(np.sum(gW ** 2))) for gW, _ in g]       # WHAT: size of each layer's weight-gradient (first layer first)

print("size of the gradient at each layer at the start (6 hidden layers of 8; layer 1 = nearest the input)")
print("  layer:      " + "".join(f"{i + 1:>11d}" for i in range(7)))
grad_sizes = {}
for act in ("sigmoid", "tanh", "relu"):
    grad_sizes[act] = layer_grad_sizes(act)
    print(f"  {act:8s}    " + "".join(f"{v:11.2e}" for v in grad_sizes[act]))
print()
print("input-side gradient / output-side gradient:  " + "   ".join(f"{a}: {grad_sizes[a][0] / grad_sizes[a][-1]:.1e}" for a in grad_sizes))

size of the gradient at each layer at the start (6 hidden layers of 8; layer 1 = nearest the input)
  layer:                1          2          3          4          5          6          7
  sigmoid        1.83e-06   1.79e-05   8.84e-05   4.52e-04   1.56e-03   9.75e-03   3.17e-02
  tanh           2.73e-03   4.78e-03   7.31e-03   9.13e-03   8.98e-03   1.28e-02   8.99e-03
  relu           3.78e-02   1.31e-01   1.53e-01   1.09e-01   3.88e-02   7.17e-02   4.36e-02

input-side gradient / output-side gradient:  sigmoid: 5.8e-05   tanh: 3.0e-01   relu: 8.7e-01


**What to notice.**
- On this easy problem **all three activations solved XOR from all 10 starts**, with small differences (relu lowest median loss, sigmoid highest). Because I used a different learning rate for each, don't read this as a ranking. The honest summary: activation barely matters on a shallow network for an easy problem.
- **Depth is where they separate.** In a 6-hidden-layer network at the start, the sigmoid net's gradient is about 17,000 times smaller at the input side than at the output side (ratio 5.8e-05), because each layer multiplies the blame by a sigmoid slope of at most 0.25. The earliest layers receive almost no learning signal. This is the **vanishing gradient** problem. Tanh (ratio 0.3) and relu (ratio 0.87) keep the signal much more alive. That is the main reason relu became the default for deep networks.
- Caveat: this is one random start and the network is untrained, so it shows the *starting* behavior, not a full training comparison.

# Part 8: Overfitting, in a network

> **You are here:** Notebook 3's lesson, now with a model flexible enough to bite. A wide network on a *small, noisy* data set.

Data: points inside vs outside a circle (the "ring" from Notebook 2), but **10% of the labels are flipped at random** (a mislabeled part, a sensor glitch). Only 60 training points (this particular draw happened to flip just 2 of the 60). The network has 32 hidden neurons: far more flexibility than 60 noisy points can justify. Watch train and validation loss.

In [13]:
def make_ring_noisy(n, seed, flip=0.10):
    r = np.random.default_rng(seed)
    X = r.uniform(-2, 2, (n, 2))
    y = (np.sum(X ** 2, axis=1) < 1.5).astype(float)
    flips = r.random(n) < flip
    return X, np.where(flips, 1 - y, y)                     # WHAT: a random 10% of the labels are wrong

Xr_tr, yr_tr = make_ring_noisy(60, seed=1); Xr_va, yr_va = make_ring_noisy(600, seed=2); Xr_te, yr_te = make_ring_noisy(600, seed=3)
print("labels flipped in training set:", int(np.sum((np.sum(Xr_tr ** 2, axis=1) < 1.5) != (yr_tr == 1))), "of 60")

pred_big, _, hist_big = train_net(Xr_tr, yr_tr, (32,), "tanh", lr=0.3, steps=20000, seed=0, Xval=Xr_va, yval=yr_va, every=100)
pred_wd, _, hist_wd = train_net(Xr_tr, yr_tr, (32,), "tanh", lr=0.3, steps=20000, seed=0, l2=0.003, Xval=Xr_va, yval=yr_va, every=100)
hb, hw = np.array(hist_big), np.array(hist_wd)
i_best = int(np.argmin(hb[:, 2]))
print(" step  | train loss | validation loss    (no leash)")
for s in (0, 100, 500, 2000, 5000, 10000, 20000):
    i = int(np.where(hb[:, 0] == s)[0][0]); print(f" {s:5d} |  {hb[i, 1]:.3f}     |  {hb[i, 2]:.3f}")
print(f"validation loss is lowest at step {int(hb[i_best, 0])} ({hb[i_best, 2]:.3f}); at step 20000 it is {hb[-1, 2]:.3f}")
print(f"with the weight leash (l2 = 0.003), step 20000: train {hw[-1, 1]:.3f}, validation {hw[-1, 2]:.3f}")
print(f"held-out test accuracy at step 20000: no leash {acc_of(pred_big, Xr_te, yr_te):.1%}, with leash {acc_of(pred_wd, Xr_te, yr_te):.1%}   (best possible is about 90% because 10% of labels are noise)")

labels flipped in training set: 2 of 60


 step  | train loss | validation loss    (no leash)
     0 |  0.698     |  0.721
   100 |  0.536     |  0.599
   500 |  0.198     |  0.457
  2000 |  0.074     |  0.728
  5000 |  0.026     |  1.071
 10000 |  0.008     |  1.332
 20000 |  0.003     |  1.532
validation loss is lowest at step 400 (0.444); at step 20000 it is 1.532
with the weight leash (l2 = 0.003), step 20000: train 0.181, validation 0.460
held-out test accuracy at step 20000: no leash 81.5%, with leash 84.2%   (best possible is about 90% because 10% of labels are noise)


In [14]:
fig, (a1_, a2_) = plt.subplots(1, 2, figsize=(11.5, 4.0))
a1_.plot(hb[1:, 0], hb[1:, 1], color=BLUE); a1_.plot(hb[1:, 0], hb[1:, 2], color=ORANGE)
a1_.axvline(hb[i_best, 0], color="#898781", linestyle="--", linewidth=1.2)
a1_.set_xscale("log"); a1_.set_xlabel("step (log scale)"); a1_.set_ylabel("loss")
tag(a1_, hb[-1, 0], hb[-1, 1], "train", BLUE, dx=5); tag(a1_, hb[-1, 0], hb[-1, 2], "validation", ORANGE, dx=5)
a1_.set_xlim(80, 1.2e5); a1_.set_title("Train keeps falling; validation turns up (dashed = best step)")
a2_.plot(hw[1:, 0], hw[1:, 1], color=BLUE); a2_.plot(hw[1:, 0], hw[1:, 2], color=ORANGE)
a2_.set_xscale("log"); a2_.set_xlabel("step (log scale)"); a2_.set_ylabel("loss")
tag(a2_, hw[-1, 0], hw[-1, 1], "train", BLUE, dx=5); tag(a2_, hw[-1, 0], hw[-1, 2], "validation", ORANGE, dx=5)
a2_.set_xlim(80, 1.2e5); a2_.set_ylim(a1_.get_ylim()); a2_.set_title("Same network with the weight leash (l2 = 0.003)")
plt.tight_layout(); plt.show()

<Figure size 1265x440 with 2 Axes>

**What to notice.**
- Without a leash, **training loss falls to 0.003** (the network has essentially memorized all 60 labels, including any flipped ones), while **validation loss is lowest at step 400 (0.444) and then climbs to 1.53**. This is Notebook 3's picture again, now in a network.
- With the weight leash (`l2 = 0.003`), validation loss stays near 0.46 for the whole run and training loss stops at 0.18: the leash stops the network from memorizing. Early stopping at step 400 would also have worked.
- Held-out accuracy was 81.5% without the leash and 84.2% with it. The gap is suggestive, not conclusive: it is one seed, one training set, and 600 test points (a difference of that size is roughly one to two standard errors).
- The same three leashes from Notebook 3 apply: more data, a smaller model or weight decay, and stopping early by watching validation.

# Part 9: The bridge to Keras and RL (and "1 billion parameters")

> **You are here:** everything you wrote maps one-to-one onto the Keras calls in the CartPole and Qmaze notebooks.

| Keras | What you built here |
|---|---|
| `Dense(h, activation="relu")` | `W` (inputs x h), `b` (h), then the squash: one entry of `net` |
| `model(x)` / `predict` | `net_forward` |
| the loss (`categorical_crossentropy`, `mse`) | `ce_loss`, or squared error for a value network |
| `model.train_on_batch(x, y)` | `net_forward` + `net_backward` + one gradient step on **every** weight |
| `optimizer` (Adam, SGD) | the update rule. SGD is Notebook 1's line; Adam adds per-weight adaptive step sizes |
| `learning_rate` | the same `lr` from Notebook 1, same speed-limit logic |
| `model.fit(..., epochs, batch_size, validation_split)` | the loop in `train_net`, with batches (Notebook 1, Part 7) and the validation pile (Notebook 3) |
| `kernel_regularizer=l2(...)` | the `l2` leash on the weights |

In the Qmaze DQN the picture went in, one Q-value per move came out, and `train_on_batch` moved **all** the weights a little toward the target, which is Notebook 1's Part 8: a Q-update as one gradient step, applied to every weight at once. In A2C the actor's last layer is the Notebook 2 neuron (sigmoid or its multi-action cousin, softmax) and the critic's last layer is a plain number.

**Counting parameters** (the "1 billion parameters" question): each layer has `inputs x outputs` weights plus `outputs` biases. Here is `count_params` on three networks:

In [15]:
examples = {
    "Part 4's XOR net (2-4-1)": [2, 4, 1],
    "a CartPole-sized net (4 inputs, two layers of 64, 2 outputs)": [4, 64, 64, 2],
    "one wide layer pair (4096 -> 4096 -> 4096)": [4096, 4096, 4096],
}
for name, sizes in examples.items():
    print(f"{name:62s} {count_params(sizes):>14,d} parameters")
per_layer = count_params([4096, 4096])
print()
print(f"one 4096-wide layer has {per_layer:,d} parameters; about {1e9 / per_layer:.0f} such layers in a stack make 1 billion.")
print("'1 billion parameters' means 1 billion of these adjustable numbers (the weights and biases), each one moved by its own slope in the backward pass.")

Part 4's XOR net (2-4-1)                                                   17 parameters
a CartPole-sized net (4 inputs, two layers of 64, 2 outputs)            4,610 parameters
one wide layer pair (4096 -> 4096 -> 4096)                         33,562,624 parameters

one 4096-wide layer has 16,781,312 parameters; about 60 such layers in a stack make 1 billion.
'1 billion parameters' means 1 billion of these adjustable numbers (the weights and biases), each one moved by its own slope in the backward pass.


So the number is just the sum of the layer-by-layer counts. Large models have many wide layers (real ones use more elaborate layers than plain `Dense`, but the counting is the same idea), and the training loop is still *forward, backward, update*. Tuning-wise, every parameter is learned, but the layer sizes, depth, activation, learning rate and so on are the knobs *you* set (the ledger below).

# Part 10: Sandbox (your ideas compete here)

> **You are here:** pick an architecture and see which problems it can handle.

**The ruler.** A classifier is a function `clf(Xtrain, ytrain, Xtest)` returning the probability of class 1. Scored by held-out log-loss and accuracy on three problems: **xor** (opposite corners), **ring** (inside a circle), and a **checkerboard** (alternating squares, the hardest: it needs many bends). 300 training points each, no label noise. Reference entries: the plain one-neuron model from Notebook 2, the hand-made-features version from Notebook 2, and three small networks.

In [16]:
def make_ring(n, seed):
    r = np.random.default_rng(seed); X = r.uniform(-2, 2, (n, 2))
    return X, (np.sum(X ** 2, axis=1) < 1.5).astype(float)

def make_checker(n, seed):
    r = np.random.default_rng(seed); X = r.uniform(-2, 2, (n, 2))
    return X, ((np.floor(X[:, 0]) + np.floor(X[:, 1])) % 2 == 0).astype(float)     # WHAT: alternating 1x1 squares

problems4 = [(nm, f(300, 100 + i), f(1000, 200 + i)) for i, (nm, f) in enumerate([("xor", make_xor), ("ring", make_ring), ("checkerboard", make_checker)])]

def score4(clf):
    out = []
    for _, (Xa, ya), (Xb, yb) in problems4:
        p = np.clip(clf(Xa, ya, Xb), 1e-9, 1 - 1e-9)
        out.append((float(-np.mean(yb * np.log(p) + (1 - yb) * np.log(1 - p))), float(np.mean((p > 0.5) == (yb == 1)))))
    return out

def race4(entries):
    print(f"{'':36s}" + "".join(f"{nm:>22s}" for nm, _, _ in problems4))
    print(f"{'':36s}" + "".join(f"{'logloss / accuracy':>22s}" for _ in problems4))
    for label, clf in entries:
        res = score4(clf)
        print(f"{label:36s}" + "".join(f"{f'{l:.3f} / {a:.0%}':>22s}" for l, a in res))

def neural(hidden, act="tanh", lr=0.5, steps=4000, l2=0.0):
    return lambda Xa, ya, Xb: train_net(Xa, ya, hidden, act, lr=lr, steps=steps, l2=l2, seed=0)[0](Xb)

def one_neuron(Xa, ya, Xb):
    return train_net(Xa, ya, (), "tanh", lr=0.5, steps=3000)[0](Xb)          # WHAT: no hidden layer = Notebook 2's single neuron

def neuron_plus_features(Xa, ya, Xb):
    feats = lambda X: np.column_stack([X, X[:, 0] * X[:, 1], X ** 2])         # WHAT: Notebook 2's hand-made inputs
    return train_net(feats(Xa), ya, (), "tanh", lr=0.5, steps=3000)[0](feats(Xb))

reference4 = [("one neuron (Notebook 2)", one_neuron),
              ("one neuron + hand-made inputs", neuron_plus_features),
              ("network 2-4-1 (tanh)", neural((4,))),
              ("network 2-16-1 (tanh)", neural((16,))),
              ("network 2-16-16-1 (tanh)", neural((16, 16), steps=6000))]
race4(reference4)

                                                       xor                  ring          checkerboard
                                        logloss / accuracy    logloss / accuracy    logloss / accuracy


one neuron (Notebook 2)                        0.704 / 39%           0.622 / 69%           0.696 / 47%


one neuron + hand-made inputs                  0.081 / 98%           0.051 / 99%           0.690 / 56%


network 2-4-1 (tanh)                           0.073 / 98%           0.085 / 96%           0.581 / 66%


network 2-16-1 (tanh)                          0.062 / 97%           0.044 / 99%           0.326 / 88%


network 2-16-16-1 (tanh)                       0.065 / 98%           0.030 / 99%           0.249 / 93%


**What to notice.**
- **One neuron** fails on xor and checkerboard (below 50% on held-out data just means it learned nothing real) and manages 69% on the ring. **One neuron plus Notebook 2's hand-made inputs** nails xor and ring (98% and 99%) but not the checkerboard (56%): those inputs don't include the right shapes for it.
- **Networks need no hand-made inputs.** The 2-4-1 network gets xor and ring but only 66% on the checkerboard. The **2-16-1** network reaches 88% and the **2-16-16-1** network reaches 93%: more squares need more bends, and wider or deeper networks provide them.
- Caveats: each entry is trained from **one** random start and tested on one data set, so differences of a few points aren't reliable (Part 5 showed how much a start can matter); the step counts and learning rates are my picks, not tuned.

### Your slot
Edit `my_net` and re-run. Ideas: other widths and depths; relu vs tanh; a different learning rate or step count; weight decay `l2`; or something of your own (an ensemble: average several networks from different random starts). Rule: only the training points may decide anything.

In [17]:
def my_net(Xa, ya, Xb):
    # SANDBOX: replace with your own idea. Return p(class 1) for every row of Xb. Starts as a small tanh network.
    return train_net(Xa, ya, (8,), "tanh", lr=0.5, steps=4000, seed=0)[0](Xb)

race4(reference4 + [("MY NET", my_net)])

                                                       xor                  ring          checkerboard
                                        logloss / accuracy    logloss / accuracy    logloss / accuracy


one neuron (Notebook 2)                        0.704 / 39%           0.622 / 69%           0.696 / 47%


one neuron + hand-made inputs                  0.081 / 98%           0.051 / 99%           0.690 / 56%


network 2-4-1 (tanh)                           0.073 / 98%           0.085 / 96%           0.581 / 66%


network 2-16-1 (tanh)                          0.062 / 97%           0.044 / 99%           0.326 / 88%


network 2-16-16-1 (tanh)                       0.065 / 98%           0.030 / 99%           0.249 / 93%


MY NET                                         0.064 / 98%           0.058 / 98%           0.495 / 76%


# Part 11: The knob ledger, self-check, and what's next

## Knob ledger (Notebooks 1 to 4)
Families: **objective**, **model**, **search**, **experience**, **judging**. New here: width, depth, activation, initialization, and the weight leash in a network.

In [18]:
rows = [
    ("learning rate (NB1)", "search", "too small: slow", "too big: blows up"),
    ("input scaling (NB1)", "experience", "unscaled: far slower", "standardized: fast and safe"),
    ("loss for categories (NB2)", "objective", "squared error stalls", "cross-entropy keeps a slope"),
    ("model size / leash / early stop (NB3)", "model, objective, judging", "too small: underfits", "too big and unleashed: overfits"),
    ("hidden width", "model", f"1 neuron: solved {sum(a >= 0.95 for a in width_results[1])}/10 starts", f"16 neurons: solved {sum(a >= 0.95 for a in width_results[16])}/10"),
    ("random start (seed)", "search / experience", "one start can fail while another succeeds", "judge a design on several starts"),
    ("init scale", "search", f"0.01: median held-out {scale_results[0.01][1]:.0%}", f"10: median held-out {scale_results[10.0][1]:.0%}  (1.0: {scale_results[1.0][1]:.0%})"),
    ("activation", "model", f"sigmoid: {act_results['sigmoid'][2]}/10 solved", f"relu: {act_results['relu'][2]}/10, tanh: {act_results['tanh'][2]}/10"),
    ("depth", "model", "see Part 7: gradient size per layer", f"input/output gradient ratio, sigmoid {grad_sizes['sigmoid'][0] / grad_sizes['sigmoid'][-1]:.0e} vs relu {grad_sizes['relu'][0] / grad_sizes['relu'][-1]:.0e}"),
    ("weight decay l2", "objective", f"no leash: val loss {hb[-1, 2]:.2f} at step 20000", f"l2 = 0.003: val loss {hw[-1, 2]:.2f}"),
]
print(f"{'knob':38s}{'family':26s}{'at one extreme':48s}{'at the other'}")
print("-" * 175)
for knob, fam, a, b in rows:
    print(f"{knob:38s}{fam:26s}{a:48s}{b}")

knob                                  family                    at one extreme                                  at the other
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
learning rate (NB1)                   search                    too small: slow                                 too big: blows up
input scaling (NB1)                   experience                unscaled: far slower                            standardized: fast and safe
loss for categories (NB2)             objective                 squared error stalls                            cross-entropy keeps a slope
model size / leash / early stop (NB3) model, objective, judging too small: underfits                            too big and unleashed: overfits
hidden width                          model                     1 neuron: solved 0/10 starts                    16 neurons: solved 10/10
random s

## Self-check (answer in your own words before you read the answers)
| Question | Short answer |
|---|---|
| Why does a hidden layer fix XOR? | It re-describes each part with new inputs (like "both high", "both low") in which a straight line works. |
| What is backpropagation in one sentence? | The chain rule, walking backward from the loss: each layer passes blame back along its weights, scaled by the squash's slope, and each weight's slope is (blame) x (the input it saw). |
| How do you know a backward pass is right? | Nudge every weight and compare. Any disagreement means a bug. |
| Why can't all weights start at zero? | Every hidden neuron would compute and receive the same thing, so they'd never differ. |
| Why can the same network succeed from one start and fail from another? | The loss has plateaus and bad regions, and the random start decides which one gradient descent falls into. |
| What does "1 billion parameters" count? | Every weight and bias: for each layer, inputs x outputs + outputs. |
| What is `train_on_batch`? | Forward pass, backward pass, and one update to every weight. |

## What's next
- You can now read any Keras `Dense` model as `W`, `b`, a squash, and a loss, and you know the backward pass isn't magic.
- You've seen overfitting recur in a network, and the same three leashes work.

**Notebook 5: probability basics.** RL is built on *expected* outcomes, and policies are *probability distributions over actions*. We'll build expectation, variance, sampling, and why averages of random things behave (law of large numbers), so that "return", "advantage" and "entropy" stop being vocabulary and become things you can compute.